# Optimization Practice Problems


## Setup code


In [ ]:
import sys
from pathlib import Path


def find_practice_root() -> Path:
    """Find Arena_Practice_Basics whether Jupyter starts here or in a parent directory."""
    for path in [Path.cwd(), *Path.cwd().parents]:
        if path.name == "Arena_Practice_Basics" and (path / "03_optimization").exists():
            return path
        candidate = path / "Arena_Practice_Basics"
        if (candidate / "03_optimization").exists():
            return candidate
    raise FileNotFoundError("Could not find Arena_Practice_Basics")


practice_root = find_practice_root()
section_dir = practice_root / "03_optimization"
cnn_dir = practice_root / "02_cnns_resnets"
source_exercises_dir = practice_root.parent / "chapter0_fundamentals" / "exercises"

# Local project modules. `source_exercises_dir` is exposed only because the unchanged
# optimization tests import `part3_optimization.solutions` for a test fixture.
if str(section_dir) not in sys.path:
    sys.path.insert(0, str(section_dir))
if str(cnn_dir) not in sys.path:
    sys.path.append(str(cnn_dir))
if str(source_exercises_dir) not in sys.path:
    sys.path.append(str(source_exercises_dir))

print(f"Practice root: {practice_root}")
print(f"Section files: {section_dir}")

import os
import time
from dataclasses import dataclass
from typing import Callable, Iterable, Literal

import numpy as np
import torch as t
import torch.distributed as dist
import torch.multiprocessing as mp
import torch.nn.functional as F
import wandb
from IPython.core.display import HTML
from IPython.display import display
from jaxtyping import Float, Int
from torch import Tensor, optim
from torch.utils.data import DataLoader, DistributedSampler
from torchvision import datasets, transforms
from tqdm import tqdm

import tests
from utils import plot_fn, plot_fn_with_points
from resnet_model import Linear, ResNet34, get_resnet_for_feature_extraction
from plotly_utils import bar, imshow, line

# Later data downloads go into Arena_Practice_Basics/data.
exercises_dir = practice_root
device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

# The distributed examples are designed to be copied to a Python file and run on a multi-GPU host.
# Keeping this false prevents notebook cells from spawning processes accidentally.
MAIN = False

print(f"Using {device=}")

WORLD_SIZE = min(t.cuda.device_count(), 3)


In [ ]:
def pathological_curve_loss(x: Tensor, y: Tensor):
    # Example of a pathological curvature. There are many more possible, feel free to experiment here!
    x_loss = t.tanh(x) ** 2 + 0.01 * t.abs(x)
    y_loss = t.sigmoid(y)
    return x_loss + y_loss


plot_fn(pathological_curve_loss, min_points=[(0, "y_min")])

### Exercise - implement `opt_fn_with_sgd`

Use `torch.optim.SGD` to minimize `fn(x, y)` from the initial point `xy`. Return the initial point
and a detached copy after every update, giving an output of shape `(n_iters + 1, 2)`.


In [ ]:
def opt_fn_with_sgd(
    fn: Callable,
    xy: Float[Tensor, "2"],
    lr: float = 0.001,
    momentum: float = 0.98,
    n_iters: int = 100,
) -> Float[Tensor, "n_iters_plus_one 2"]:
    """Return every (x, y) point, including the initial point and `n_iters` updates."""
    assert xy.requires_grad
    
    optimizer = optim.SGD((xy,), lr=lr, momentum=momentum)

    xy_list = [xy.detach().clone()]  # so we don't unintentionally modify past values in `xy_list`

    for i in range(n_iters):
        fn(xy[0], xy[1]).backward()
        optimizer.step()
        optimizer.zero_grad()
        xy_list.append(xy.detach().clone())

    return t.stack(xy_list)


points = []
for params in [{"lr": 0.1, "momentum": 0.0}, {"lr": 0.02, "momentum": 0.99}]:
    xy = t.tensor([2.5, 2.5], requires_grad=True)
    xys = opt_fn_with_sgd(pathological_curve_loss, xy, **params)
    points.append((xys, optim.SGD, params))
    print(f"{params=}, last point={xys[-1]}")

plot_fn_with_points(pathological_curve_loss, points=points, min_points=[(0, "y_min")])

### Exercise - implement `SGD`

Complete `step`. For each parameter, apply weight decay and momentum before the update:

\[
g_t \leftarrow g_t + \lambda\theta_{t-1},\qquad
b_t \leftarrow \mu b_{t-1}+g_t,\qquad
\theta_t \leftarrow \theta_{t-1}-\gamma b_t.
\]

Skip the weight-decay or momentum operation when its coefficient is zero.


In [ ]:
class SGD:
    def __init__(
        self,
        params: Iterable[t.nn.Parameter],
        lr: float,
        momentum: float = 0.0,
        weight_decay: float = 0.0,
    ):
        self.params = list(params)
        self.lr = lr
        self.mu = momentum
        self.lmda = weight_decay
        self.b = [t.zeros_like(p) for p in self.params]

    def zero_grad(self) -> None:
        for param in self.params:
            param.grad = None

    @t.inference_mode()
    def step(self) -> None:
        for b, theta in zip(self.b, self.params):
            g = theta.grad
            if self.lmda != 0:
                g = g + self.lmda * theta  # this shouldn't be inplace since we don't want to modify theta.grad
            if self.mu != 0:
                b.copy_(self.mu * b + g)  # this does need to be inplace, since we're modifying the value in `self.b`
                g = b
            theta -= self.lr * g  # inplace operation, to modify params


    def __repr__(self) -> str:
        return f"SGD(lr={self.lr}, momentum={self.mu}, weight_decay={self.lmda})"


tests.test_sgd(SGD)

### Exercise - implement `RMSprop`

Complete `step` using:

\[
v_t \leftarrow \alpha v_{t-1}+(1-\alpha)g_t^2,\qquad
g_t \leftarrow \frac{g_t}{\sqrt{v_t}+\epsilon}.
\]

Apply weight decay before the running variance and momentum after normalization.


In [ ]:
class RMSprop:
    def __init__(
        self,
        params: Iterable[t.nn.Parameter],
        lr: float = 0.01,
        alpha: float = 0.99,
        eps: float = 1e-8,
        weight_decay: float = 0.0,
        momentum: float = 0.0,
    ):
        self.params = list(params)
        self.lr = lr
        self.eps = eps
        self.mu = momentum
        self.lmda = weight_decay
        self.alpha = alpha
        self.b = [t.zeros_like(p) for p in self.params]
        self.v = [t.zeros_like(p) for p in self.params]

    def zero_grad(self) -> None:
        for param in self.params:
            param.grad = None

    @t.inference_mode()
    def step(self) -> None:
        for theta, b, v in zip(self.params, self.b, self.v):
            g = theta.grad
            if self.lmda != 0:
                g = g + self.lmda * theta
            v.copy_(self.alpha * v + (1 - self.alpha) * g.pow(2))  # inplace operation, to modify value in self.v
            g = g / (v.sqrt() + self.eps)  # not inplace operation
            if self.mu > 0:
                b.copy_(self.mu * b + g)  # inplace operation, to modify value in self.b
                g = b
            theta -= self.lr * g  # inplace operation, to modify params


    def __repr__(self) -> str:
        return f"RMSprop(lr={self.lr}, alpha={self.alpha}, eps={self.eps}, momentum={self.mu}, weight_decay={self.lmda})"


tests.test_rmsprop(RMSprop)

### Exercise - implement `Adam`

Complete `step`. Track first and second moments, bias-correct both, update each parameter, and
increment `self.t` once per optimizer step:

\[
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t,\quad
v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2.
\]


In [ ]:
class Adam:
    def __init__(
        self,
        params: Iterable[t.nn.Parameter],
        lr: float = 0.001,
        betas: tuple[float, float] = (0.9, 0.999),
        eps: float = 1e-8,
        weight_decay: float = 0.0,
    ):
        self.params = list(params)
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.lmda = weight_decay
        self.t = 1
        self.m = [t.zeros_like(p) for p in self.params]
        self.v = [t.zeros_like(p) for p in self.params]

    def zero_grad(self) -> None:
        for param in self.params:
            param.grad = None

    @t.inference_mode()
    def step(self) -> None:
        for theta, m, v in zip(self.params, self.m, self.v):
            g = theta.grad
            if self.lmda != 0:
                g = g + self.lmda * theta
            m.copy_(self.beta1 * m + (1 - self.beta1) * g)
            v.copy_(self.beta2 * v + (1 - self.beta2) * g.pow(2))
            m_hat = m / (1 - self.beta1**self.t)
            v_hat = v / (1 - self.beta2**self.t)
            theta -= self.lr * m_hat / (v_hat.sqrt() + self.eps)
        self.t += 1

        

    def __repr__(self) -> str:
        return f"Adam(lr={self.lr}, betas=({self.beta1}, {self.beta2}), eps={self.eps}, weight_decay={self.lmda})"


tests.test_adam(Adam)

### Exercise - implement `AdamW`

Adapt Adam so weight decay is applied directly to each parameter rather than added to its gradient.
The moment calculations should use the original gradient.


In [ ]:
class AdamW:
    def __init__(
        self,
        params: Iterable[t.nn.Parameter],
        lr: float = 0.001,
        betas: tuple[float, float] = (0.9, 0.999),
        eps: float = 1e-8,
        weight_decay: float = 0.0,
    ):
        self.params = list(params)
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.lmda = weight_decay
        self.t = 1
        self.m = [t.zeros_like(p) for p in self.params]
        self.v = [t.zeros_like(p) for p in self.params]

    def zero_grad(self) -> None:
        for param in self.params:
            param.grad = None

    @t.inference_mode()
    def step(self) -> None:
        for theta, m, v in zip(self.params, self.m, self.v):
            g = theta.grad
            theta *= 1 - self.lr * self.lmda
            m.copy_(self.beta1 * m + (1 - self.beta1) * g)
            v.copy_(self.beta2 * v + (1 - self.beta2) * g.pow(2))
            m_hat = m / (1 - self.beta1**self.t)
            v_hat = v / (1 - self.beta2**self.t)
            theta -= self.lr * m_hat / (v_hat.sqrt() + self.eps)
        self.t += 1

    def __repr__(self) -> str:
        return f"AdamW(lr={self.lr}, betas=({self.beta1}, {self.beta2}), eps={self.eps}, weight_decay={self.lmda})"


tests.test_adamw(AdamW)

### Exercise - experiment with different optimizers and parameters

Run SGD, RMSprop, Adam, and AdamW on the supplied loss landscapes. Change their hyperparameters and
compare the paths they take, their stability, and how closely they approach a minimum.


In [ ]:
def opt_fn(
    fn: Callable,
    xy: Tensor,
    optimizer_class,
    optimizer_hyperparams: dict,
    n_iters: int = 100,
) -> Tensor:
    """Optimize the a given function starting from the specified point.

    optimizer_class: one of the optimizers you've defined, either SGD, RMSprop, or Adam
    optimzer_kwargs: keyword arguments passed to your optimiser (e.g. lr and weight_decay)
    """
    assert xy.requires_grad

    optimizer = optimizer_class([xy], **optimizer_hyperparams)

    xy_list = [xy.detach().clone()]  # so that we don't unintentionally modify past values in `xy_list`

    for i in range(n_iters):
        fn(xy[0], xy[1]).backward()
        optimizer.step()
        optimizer.zero_grad()
        xy_list.append(xy.detach().clone())

    return t.stack(xy_list)


points = []

optimizer_list = [
    (SGD, {"lr": 0.03, "momentum": 0.99}),
    (RMSprop, {"lr": 0.02, "alpha": 0.99, "momentum": 0.8}),
    (Adam, {"lr": 0.2, "betas": (0.99, 0.99), "weight_decay": 0.005}),
    (AdamW, {"lr": 0.2, "betas": (0.99, 0.99), "weight_decay": 0.005}),
]

for optimizer_class, params in optimizer_list:
    xy = t.tensor([2.5, 2.5], requires_grad=True)
    xys = opt_fn(
        pathological_curve_loss,
        xy=xy,
        optimizer_class=optimizer_class,
        optimizer_hyperparams=params,
    )
    points.append((xys, optimizer_class, params))

plot_fn_with_points(pathological_curve_loss, min_points=[(0, "y_min")], points=points)

In [ ]:
def bivariate_gaussian(x, y, x_mean=0.0, y_mean=0.0, x_sig=1.0, y_sig=1.0):
    norm = 1 / (2 * np.pi * x_sig * y_sig)
    x_exp = 0.5 * ((x - x_mean) ** 2) / (x_sig**2)
    y_exp = 0.5 * ((y - y_mean) ** 2) / (y_sig**2)
    return norm * t.exp(-x_exp - y_exp)


means = [(1.0, -0.5), (-1.0, 0.5), (-0.5, -0.8)]


def neg_trimodal_func(x, y):
    """
    This function has 3 global minima, at `means`. Unstable methods can overshoot these minima, and
    non-adaptive methods can fail to converge to them in the first place given how shallow the
    gradients are everywhere except in the close vicinity of the minima.
    """
    z = -bivariate_gaussian(x, y, x_mean=means[0][0], y_mean=means[0][1], x_sig=0.2, y_sig=0.2)
    z -= bivariate_gaussian(x, y, x_mean=means[1][0], y_mean=means[1][1], x_sig=0.2, y_sig=0.2)
    z -= bivariate_gaussian(x, y, x_mean=means[2][0], y_mean=means[2][1], x_sig=0.2, y_sig=0.2)
    return z


plot_fn(neg_trimodal_func, x_range=(-2, 2), y_range=(-2, 2), min_points=means)

In [ ]:
def rosenbrocks_banana_func(x: Tensor, y: Tensor, a=1, b=100) -> Tensor:
    """
    This function has a global minimum at `(a, a)` so in this case `(1, 1)`. It's characterized by a
    long, narrow, parabolic valley (parameterized by `y = x**2`). Various gradient descent methods
    have trouble navigating this valley because they often oscillate unstably (gradients from the
    `b`-term dwarf the gradients from the `a`-term).

    See more on this function: https://en.wikipedia.org/wiki/Rosenbrock_function.
    """
    return (a - x) ** 2 + b * (y - x**2) ** 2 + 1


plot_fn(
    rosenbrocks_banana_func,
    x_range=(-2.5, 2.5),
    y_range=(-2, 4),
    z_range=(0, 100),
    min_points=[(1, 1)],
)

### Exercise - rewrite `SGD` to use parameter groups

Complete all three methods. Each entry in `self.param_groups` should contain its parameters and
effective hyperparameters, with precedence: group value → constructor keyword → default value.
Use the group-specific values in `step`.


In [ ]:
class SGD:
    def __init__(self, params, **kwargs):
        """Implements SGD with momentum.

        Accepts parameters in groups, or an iterable.

        Like the PyTorch version, but assume nesterov=False, maximize=False, and dampening=0
            https://pytorch.org/docs/stable/generated/torch.optim.SGD.html#torch.optim.SGD
        """
        # Deal with case where we didn't supply groups, so we just make it into a single dictionary
        if not isinstance(params, (list, tuple)):
            params = [{"params": params}]

        # Make sure each group["params"] is a list of params not a generator (so we don't iterate
        # over & destroy it!)
        for p in params:
            p["params"] = list(p["params"])

        self.param_groups = []

        # YOUR CODE HERE - fill in `self.param_groups`
        raise NotImplementedError()

    def zero_grad(self) -> None:
        raise NotImplementedError()

    @t.inference_mode()
    def step(self) -> None:
        raise NotImplementedError()


tests.test_sgd_param_groups(SGD)

## Weights & Biases exercises


In [ ]:
def get_cifar() -> tuple[datasets.CIFAR10, datasets.CIFAR10]:
    """Returns the locally installed CIFAR-10 train and test sets."""
    data_dir = exercises_dir / "data"
    cifar_trainset = datasets.CIFAR10(data_dir, train=True, download=False, transform=IMAGENET_TRANSFORM)
    cifar_testset = datasets.CIFAR10(data_dir, train=False, download=False, transform=IMAGENET_TRANSFORM)
    return cifar_trainset, cifar_testset


IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

IMAGENET_TRANSFORM = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)


In [ ]:
@dataclass
class ResNetFinetuningArgs:
    n_classes: int = 10
    batch_size: int = 128
    epochs: int = 3
    learning_rate: float = 1e-3
    weight_decay: float = 0.0


class ResNetFinetuner:
    def __init__(self, args: ResNetFinetuningArgs):
        self.args = args

    def pre_training_setup(self):
        self.model = get_resnet_for_feature_extraction(self.args.n_classes).to(device)
        self.optimizer = AdamW(
            self.model.out_layers[-1].parameters(),
            lr=self.args.learning_rate,
            weight_decay=self.args.weight_decay,
        )
        self.trainset, self.testset = get_cifar()
        self.train_loader = DataLoader(self.trainset, batch_size=self.args.batch_size, shuffle=True)
        self.test_loader = DataLoader(self.testset, batch_size=self.args.batch_size, shuffle=False)
        self.logged_variables = {"loss": [], "accuracy": []}
        self.examples_seen = 0

    def training_step(
        self,
        imgs: Float[Tensor, "batch channels height width"],
        labels: Int[Tensor, " batch"],
    ) -> Float[Tensor, ""]:
        """Perform a gradient update step on a single batch of data."""
        imgs, labels = imgs.to(device), labels.to(device)

        logits = self.model(imgs)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        self.optimizer.step()
        self.optimizer.zero_grad()

        self.examples_seen += imgs.shape[0]
        self.logged_variables["loss"].append(loss.item())
        return loss

    @t.inference_mode()
    def evaluate(self) -> float:
        """Evaluate the model on the test set and return the accuracy."""
        self.model.eval()
        total_correct, total_samples = 0, 0

        for imgs, labels in tqdm(self.test_loader, desc="Evaluating"):
            imgs, labels = imgs.to(device), labels.to(device)
            logits = self.model(imgs)
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_samples += len(imgs)

        accuracy = total_correct / total_samples
        self.logged_variables["accuracy"].append(accuracy)
        return accuracy

    def train(self) -> dict[str, list[float]]:
        self.pre_training_setup()

        accuracy = self.evaluate()

        for epoch in range(self.args.epochs):
            self.model.train()

            pbar = tqdm(self.train_loader, desc="Training")
            for imgs, labels in pbar:
                loss = self.training_step(imgs, labels)
                pbar.set_postfix(loss=f"{loss:.3f}", ex_seen=f"{self.examples_seen:06}")

            accuracy = self.evaluate()
            pbar.set_postfix(loss=f"{loss:.3f}", accuracy=f"{accuracy:.2f}", ex_seen=f"{self.examples_seen:06}")

        return self.logged_variables

#### Optional run — baseline feature extraction

The next cell starts the original non-W&B training run. Run it only when you explicitly want to train.


In [ ]:
baseline_args = ResNetFinetuningArgs()
baseline_trainer = ResNetFinetuner(baseline_args)
baseline_logged_variables = baseline_trainer.train()

In [ ]:
# Plot an already completed baseline run; this cell does not train.
baseline_losses = baseline_logged_variables["loss"]
baseline_accuracies = baseline_logged_variables["accuracy"]

line(
    y=[baseline_losses, baseline_accuracies],
    x_max=len(baseline_losses) * baseline_args.batch_size,
    yaxis2_range=[0, 1],
    use_secondary_yaxis=True,
    labels={"x": "Examples seen", "y1": "Cross entropy loss", "y2": "Test accuracy"},
    title="Feature extraction with ResNet34",
    width=800,
)

### Exercise - rewrite the training loop with W&B

Complete the four methods. Initialize and watch the run in `pre_training_setup`, log loss and
accuracy against `examples_seen`, and call `wandb.finish()` after training.


In [ ]:
@dataclass
class WandbResNetFinetuningArgs(ResNetFinetuningArgs):
    """Contains new params for use in wandb.init, as well as all the ResNetFinetuningArgs params."""

    wandb_project: str | None = "day3-resnet"
    wandb_name: str | None = None


class WandbResNetFinetuner(ResNetFinetuner):
    args: WandbResNetFinetuningArgs  # adding this line helps with typechecker!
    examples_seen: int = 0  # tracking examples seen (used as step for wandb)

    def pre_training_setup(self):
        """Initializes the wandb run using `wandb.init` and `wandb.watch`."""
        super().pre_training_setup()
        if wandb.run is None:
            wandb.init(project=self.args.wandb_project, name=self.args.wandb_name, config=self.args)
        wandb.watch(self.model.out_layers[-1], log="all", log_freq=50)
        self.examples_seen = 0

    def training_step(
        self,
        imgs: Float[Tensor, "batch channels height width"],
        labels: Int[Tensor, " batch"],
    ) -> Float[Tensor, ""]:
        """Equivalent to ResNetFinetuner.training_step, but logging the loss to wandb."""
        imgs, labels = imgs.to(device), labels.to(device)
        logits = self.model(imgs)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        self.optimizer.step()
        self.optimizer.zero_grad()

        self.examples_seen += imgs.shape[0]
        wandb.log({"loss": loss.item()}, step=self.examples_seen)
        return loss

    @t.inference_mode()
    def evaluate(self) -> float:
        """Equivalent to ResNetFinetuner.evaluate, but logging the accuracy to wandb."""
        self.model.eval()
        total_correct, total_samples = 0, 0

        for imgs, labels in tqdm(self.test_loader, desc="Evaluating"):
            imgs, labels = imgs.to(device), labels.to(device)
            logits = self.model(imgs)
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_samples += len(imgs)

        accuracy = total_correct / total_samples
        wandb.log({"accuracy": accuracy}, step=self.examples_seen)
        return accuracy

    def train(self) -> None:
        """Equivalent to ResNetFinetuner.train, but with wandb integration."""
        self.pre_training_setup()
        try:
            self.evaluate()

            for _ in range(self.args.epochs):
                self.model.train()
                pbar = tqdm(self.train_loader, desc="Training")
                for imgs, labels in pbar:
                    loss = self.training_step(imgs, labels)
                    pbar.set_postfix(loss=f"{loss:.3f}", ex_seen=f"{self.examples_seen:06}")

                accuracy = self.evaluate()
                pbar.set_postfix(
                    loss=f"{loss:.3f}",
                    accuracy=f"{accuracy:.2f}",
                    ex_seen=f"{self.examples_seen:06}",
                )
        finally:
            wandb.finish()


#### Optional run — one W&B training run

This execution cell is isolated from the class definition. Running it starts one complete W&B training run.


In [ ]:
wandb_args = WandbResNetFinetuningArgs()
wandb_trainer = WandbResNetFinetuner(wandb_args)
wandb_trainer.train()

### Exercise - define a sweep configuration and update `args`

Configure a random sweep which maximizes accuracy. Sample learning rate log-uniformly from
`1e-4`–`1e-1`, batch size from `[32, 64, 128, 256]`, and weight decay with equal probability of
zero or a log-uniform value from `1e-4`–`1e-2`. Then implement `update_args`.


In [ ]:
sweep_config = dict(
    method="random",
    metric=dict(name="accuracy", goal="maximize"),
    parameters=dict(
        learning_rate=dict(min=1e-4, max=1e-1, distribution="log_uniform_values"),
        batch_size=dict(values=[32, 64, 128, 256]),
        weight_decay=dict(min=1e-4, max=1e-2, distribution="log_uniform_values"),
        weight_decay_bool=dict(values=[True, False]),
    ),
)


def update_args(args: WandbResNetFinetuningArgs, sampled_parameters: dict) -> WandbResNetFinetuningArgs:
    """
    Returns a new args object with modified values. The dictionary `sampled_parameters` will have
    the same keys as your `sweep_config["parameters"]` dict, and values equal to the sampled values
    of those hyperparameters.
    """
    assert set(sampled_parameters.keys()) == set(sweep_config["parameters"].keys())

    args.learning_rate = sampled_parameters["learning_rate"]
    args.batch_size = sampled_parameters["batch_size"]
    args.weight_decay = sampled_parameters["weight_decay"] if sampled_parameters["weight_decay_bool"] else 0.0
    return args


tests.test_sweep_config(sweep_config)
tests.test_update_args(update_args, sweep_config)

#### Optional run — W&B sweep

The first cell only defines one sweep trial. The second cell creates the sweep and starts trials; run it only when wanted.


In [ ]:
def train_sweep_trial():
    args = WandbResNetFinetuningArgs()
    wandb.init(project=args.wandb_project, name=args.wandb_name, reinit=False)
    try:
        args = update_args(args, dict(wandb.config))
        trainer = WandbResNetFinetuner(args)
        trainer.train()
    finally:
        if wandb.run is not None:
            wandb.finish()

In [ ]:
# This cell starts new training runs. Increase count only after one successful trial.
sweep_id = wandb.sweep(sweep=sweep_config, project="day3-resnet-sweep")
wandb.agent(sweep_id=sweep_id, function=train_sweep_trial, count=1)

## Distributed-training exercises


### Exercise - implement `broadcast`

After the function returns, every process should hold the tensor originally stored by process
`src`. These exercises require a separate Python file and a multi-GPU environment; `MAIN` is kept
false in this notebook to prevent accidental process spawning.


In [ ]:
def broadcast(tensor: Tensor, rank: int, world_size: int, src: int = 0):
    """
    Broadcast averaged gradients from rank 0 to all other ranks.
    """
    raise NotImplementedError()


if MAIN:
    tests.test_broadcast(broadcast, WORLD_SIZE)

### Exercise - implement `reduce` and `all_reduce`

`reduce` should sum or average tensors into process `dst`. `all_reduce` should perform the same
reduction and then broadcast the result to every process.


In [ ]:
def reduce(tensor, rank, world_size, dst=0, op: Literal["sum", "mean"] = "sum"):
    """
    Reduces gradients to rank `dst`, so this process contains the sum or mean of all tensors across
    processes.
    """
    raise NotImplementedError()


def all_reduce(tensor, rank, world_size, op: Literal["sum", "mean"] = "sum"):
    """
    Allreduce the tensor across all ranks, using 0 as the initial gathering rank.
    """
    raise NotImplementedError()


if MAIN:
    tests.test_reduce(reduce, WORLD_SIZE)
    tests.test_all_reduce(all_reduce, WORLD_SIZE)

### Exercise - complete `DistResNetTrainer`

Complete setup, training, evaluation, and orchestration for one trainer per GPU. Distribute the
data, synchronize gradients with `all_reduce`, and restrict W&B logging to rank zero.


In [ ]:
def get_untrained_resnet(n_classes: int) -> ResNet34:
    """
    Gets an untrained ResNet using your reusable local CNN implementation.
    """
    resnet = ResNet34()
    resnet.out_layers[-1] = Linear(resnet.out_features_per_group[-1], n_classes)
    return resnet


@dataclass
class DistResNetTrainingArgs(WandbResNetFinetuningArgs):
    world_size: int = 1
    wandb_project: str | None = "day3-resnet-dist-training"


class DistResNetTrainer:
    args: DistResNetTrainingArgs

    def __init__(self, args: DistResNetTrainingArgs, rank: int):
        self.args = args
        self.rank = rank
        self.device = t.device(f"cuda:{rank}")

    def pre_training_setup(self):
        raise NotImplementedError()

    def training_step(self, imgs: Tensor, labels: Tensor) -> Tensor:
        raise NotImplementedError()

    @t.inference_mode()
    def evaluate(self) -> float:
        raise NotImplementedError()

    def train(self):
        raise NotImplementedError()


def dist_train_resnet_from_scratch(rank, world_size):
    dist.init_process_group(backend="nccl", rank=rank, world_size=world_size)
    args = DistResNetTrainingArgs(world_size=world_size)
    trainer = DistResNetTrainer(args, rank)
    trainer.train()
    dist.destroy_process_group()


if MAIN:
    world_size = t.cuda.device_count()
    mp.spawn(
        dist_train_resnet_from_scratch,
        args=(world_size,),
        nprocs=world_size,
        join=True,
    )

### Bonus - implement ring all-reduce

Replace the single-process bottleneck with cyclic non-blocking sends and receives. Support both
sum and mean reductions and pass the supplied distributed test.


In [ ]:
def ring_all_reduce(tensor: Tensor, rank, world_size, op: Literal["sum", "mean"] = "sum") -> None:
    """
    Ring all_reduce implementation using non-blocking send/recv to avoid deadlock.
    """
    raise NotImplementedError()


if MAIN:
    tests.test_all_reduce(ring_all_reduce)